# Projet CUDA — PSO → DE massivement parallèle
## Étape 1 : faire tourner le code PSO du prof, le comprendre, et le diagnostiquer

**Avant tout :** `Exécution → Modifier le type d'exécution → GPU T4`.

**Objectif de cette étape** (≈ 2 séances) :
1. Vérifier que le GPU et `nvcc` fonctionnent sur Colab.
2. Compiler et exécuter le code `cudapso` **sans rien modifier** → c'est notre *baseline*.
3. Comprendre qui fait quoi (CPU / GPU, mémoire, boucle principale).
4. Repérer les défauts du code : on va réutiliser son squelette pour le DE, donc chaque bug qu'on laisse passer se retrouvera dans le DE.

⚠️ Les premières cellules du notebook du prof (`apt-get install cuda` 10.0 pour Ubuntu 16.04, drivers `nvidia-390`, `nvcc4jupyter`) sont **obsolètes** : Colab a déjà CUDA 12 installé. Ne les exécute pas, elles font perdre 5 minutes et peuvent casser l'environnement.

### 0 — Récupérer le dépôt GitHub dans Colab (à faire à chaque session)
Colab efface `/content` à la déconnexion : on **clone le dépôt** au début et on **pousse (`git push`)** à la fin. Rien n'est perdu tant que tu as poussé.

**Une seule fois :** dans Colab, icône 🔑 *Secrets* (barre de gauche) → *Ajouter un secret* → nom `GITHUB_TOKEN`, valeur = ton token GitHub (voir README) → active *Accès au notebook*.

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"   # ton pseudo GitHub
REPO    = "DE_CUDA"
token   = userdata.get("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO} || (cd /content/{REPO} && git pull -q)
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git log --oneline -5

In [ ]:
%cd /content/DE_CUDA/src/pso_original
!ls

In [ ]:
!nvidia-smi
!nvcc --version

### 1.1 — Écrire les fichiers du prof (copie exacte, non modifiée)
`%%writefile` crée le fichier dans le répertoire courant de Colab, pas besoin de glisser-déposer.

In [ ]:
%%writefile kernel.h
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <iostream>
#include <string>


// Constantes
/* Objective function
0: Levy 3-dimensional
1: Shifted Rastigrin's Function
2: Shifted Rosenbrock's Function
3: Shifted Griewank's Function
4: Shifted Sphere's Function
*/
const int SELECTED_OBJ_FUNC = 0;
const int NUM_OF_PARTICLES = 512;
const int NUM_OF_DIMENSIONS = 3;
const int MAX_ITER = NUM_OF_DIMENSIONS * pow(10, 4);
const float START_RANGE_MIN = -5.12f;
const float START_RANGE_MAX = 5.12f;
const float OMEGA = 0.5;
const float c1 = 1.5;
const float c2 = 1.5;
const float phi = 3.1415;

// Les 3 fonctions très utiles
float getRandom(float low, float high);
float getRandomClamped();
float host_fitness_function(float x[]);

// Fonction externe qui va tourner sur le GPU
extern "C" void cuda_pso(float *positions, float *velocities, float *pBests, float *gBest);



In [ ]:
%%writefile kernel.cu
#include <cuda_runtime.h>
#include <cuda.h>
#include <math_functions.h>

#include "kernel.h"


__device__ float tempParticle1[NUM_OF_DIMENSIONS];
__device__ float tempParticle2[NUM_OF_DIMENSIONS];

/* Objective function
0: Levy 3-dimensional
1: Shifted Rastigrin's Function
2: Shifted Rosenbrock's Function
3: Shifted Griewank's Function
4: Shifted Sphere's Function
*/
/**
 * Runs on the GPU, called from the GPU.
*/
__device__ float fitness_function(float x[]) {
    float res = 0;
    float somme = 0;
    float produit = 0;

    switch (SELECTED_OBJ_FUNC)  {
        case 0: 
            float y1 = 1 + (x[0] - 1)/4;
            float yn = 1 + (x[NUM_OF_DIMENSIONS-1] - 1)/4;

            res += pow(sin(phi*y1), 2);

            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float y = 1 + (x[i] - 1)/4;
                float yp = 1 + (x[i+1] - 1)/4;
                res += pow(y - 1, 2)*(1 + 10*pow(sin(phi*yp), 2)) + pow(yn - 1, 2);
            }
            break;
        case 1: 
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2) - 10*cos(2*phi*zi) + 10;
            }
            res -= 330;
            break;
        
        case 2:
            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float zi = x[i] - 0 + 1;
                float zip1 = x[i+1] - 0 + 1;
                res += 100 * ( pow(pow(zi, 2) - zip1, 2)) + pow(zi - 1, 2);
            }
            res += 390;
            break;
        case 3:
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                somme += pow(zi, 2)/4000;
                produit *= cos(zi/pow(i+1, 0.5));
            }
            res = somme - produit + 1 - 180; 
            break;
        case 4:
            for(int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2);
            }
            res -= 450;
            break;
    }

    return res;
}

/**
 * 
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdateParticle(float *positions, float *velocities, 
                                     float *pBests, float *gBest, float r1, 
                                     float r2)
{

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // avoid an out of bound for the array 
    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS)
        return;

    //float rp = getRandomClamped();
    //float rg = getRandomClamped();
    
    float rp = r1; // random weight for personnal =>  computed from @getRandomClamped
    float rg = r2; // random weight for global =>  computed from @getRandomClamped


    // Mise à jour de velocities et positions
    velocities[i] = OMEGA * velocities[i] + 
                    c1 * rp * (pBests[i] - positions[i]) + 
                    c2 * rg * (gBest[i % NUM_OF_DIMENSIONS] - positions[i]);

    // Update posisi particle
    //Mise à jour de la position de la particule courante
    //incrémentant la position de la particule courante avec la vitesse de la particule courante
    positions[i] += velocities[i];
}

/**
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdatePBest(float *positions, float *pBests, float* gBest)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    
    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS || i % NUM_OF_DIMENSIONS != 0)
        return;

    for (int j = 0; j < NUM_OF_DIMENSIONS; j++)
    {
        tempParticle1[j] = positions[i + j];
        tempParticle2[j] = pBests[i + j];
    }

    if (fitness_function(tempParticle1) < fitness_function(tempParticle2))
    {
        for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
            pBests[i + k] = positions[i + k];
    }
}


extern "C" void cuda_pso(float *positions, float *velocities, float *pBests, float *gBest)
{

    int size = NUM_OF_PARTICLES * NUM_OF_DIMENSIONS;
    
    // declare all the arrays on the device
    float *devPos;
    float *devVel;
    float *devPBest;
    float *devGBest;
    
    float temp[NUM_OF_DIMENSIONS];
        
    // Memory allocation
    cudaMalloc((void**)&devPos, sizeof(float) * size);
    cudaMalloc((void**)&devVel, sizeof(float) * size);
    cudaMalloc((void**)&devPBest, sizeof(float) * size);
    cudaMalloc((void**)&devGBest, sizeof(float) * NUM_OF_DIMENSIONS);
    
    // Thread & Block number
    int threadsNum = 32;
    int blocksNum = ceil(size / threadsNum);
    
    // Copy particle datas from host to device
    /**
     * Copy in GPU memory the data from the host 
     * */
    cudaMemcpy(devPos, positions, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devVel, velocities, sizeof(float) * size, 
               cudaMemcpyHostToDevice);
    cudaMemcpy(devPBest, pBests, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, 
               cudaMemcpyHostToDevice);
    
    // PSO main function
    // MAX_ITER = 30000;

    for (int iter = 0; iter < MAX_ITER; iter++)
    {     

        kernelUpdateParticle<<<blocksNum, threadsNum>>>(devPos, devVel, 
                                                        devPBest, devGBest, 
                                                        getRandomClamped(), 
                                                        getRandomClamped());  

        kernelUpdatePBest<<<blocksNum, threadsNum>>>(devPos, devPBest, 
                                                     devGBest);
        
        cudaMemcpy(pBests, devPBest, 
                   sizeof(float) * NUM_OF_PARTICLES * NUM_OF_DIMENSIONS, 
                   cudaMemcpyDeviceToHost);
        
        
        for(int i = 0; i < size; i += NUM_OF_DIMENSIONS)
        {
            for(int k = 0; k < NUM_OF_DIMENSIONS; k++) //ssB1 
                temp[k] = pBests[i + k];
        
            if (host_fitness_function(temp) < host_fitness_function(gBest))
            {
                for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
                    gBest[k] = temp[k];
            }   
        }
        
        cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, 
                   cudaMemcpyHostToDevice);
    }
    
    cudaMemcpy(positions, devPos, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(velocities, devVel, sizeof(float) * size, 
               cudaMemcpyDeviceToHost);
    cudaMemcpy(pBests, devPBest, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(gBest, devGBest, sizeof(float) * NUM_OF_DIMENSIONS, 
               cudaMemcpyDeviceToHost); 
    
    
    // cleanup
    cudaFree(devPos);
    cudaFree(devVel);
    cudaFree(devPBest);
    cudaFree(devGBest);
}



In [ ]:
%%writefile kernel.cpp
#include "kernel.h"

/* Objective function
0: Levy 3-dimensional
1: Shifted Rastigrin's Function
2: Shifted Rosenbrock's Function
3: Shifted Griewank's Function
4: Shifted Sphere's Function
*/
// parametre 1 individu avec ses positions
float host_fitness_function(float x[]) {
    float res = 0;
    float somme = 0;
    float produit = 0;

    switch (SELECTED_OBJ_FUNC)  {
        case 0: {
            float y1 = 1 + (x[0] - 1)/4;
            float yn = 1 + (x[NUM_OF_DIMENSIONS-1] - 1)/4;

            res += pow(sin(phi*y1), 2);

            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float y = 1 + (x[i] - 1)/4;
                float yp = 1 + (x[i+1] - 1)/4;
                res += pow(y - 1, 2)*(1 + 10*pow(sin(phi*yp), 2)) + pow(yn - 1, 2);
            }
            break;
        }
        case 1: {
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2) - 10*cos(2*phi*zi) + 10;
            }
            res -= 330;
            break;
        }
        case 2:
            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float zi = x[i] - 0 + 1;
                float zip1 = x[i+1] - 0 + 1;
                res += 100 * ( pow(pow(zi, 2) - zip1, 2)) + pow(zi - 1, 2);
            }
            res += 390;
            break;
        case 3:
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                somme += pow(zi, 2)/4000;
                produit *= cos(zi/pow(i+1, 0.5));
            }
            res = somme - produit + 1 - 180; 
            break;
        case 4:
            for(int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2);
            }
            res -= 450;
            break;
    }

    return res;
}

// Obtenir un random entre low et high
float getRandom(float low, float high) {
    return low + float(((high - low) + 1)*rand()/(RAND_MAX + 1.0));
}
// Obtenir un random entre 0.0f and 1.0f inclusif
float getRandomClamped() {
    return (float) rand()/(float) RAND_MAX;
}



In [ ]:
%%writefile main.cpp
#include "kernel.h"

int main(int argc, char** argv) {

    if(argc == 3){
        int dim = std::stoi(argv[1]);
        int pop = std::stoi(argv[2]);
    }
    

    float positions[NUM_OF_PARTICLES*NUM_OF_DIMENSIONS];
    float velocities[NUM_OF_PARTICLES*NUM_OF_DIMENSIONS];
    float pBests[NUM_OF_PARTICLES*NUM_OF_DIMENSIONS];
    float gBest[NUM_OF_DIMENSIONS];
    
    printf("Type \t Time \t  \t Minimum\n");
    
        // Initialisation du random
        srand((unsigned) time(NULL));

        for (int i = 0; i < NUM_OF_PARTICLES*NUM_OF_DIMENSIONS; i++) {
            positions[i] = getRandom(START_RANGE_MIN, START_RANGE_MAX);
            pBests[i] = positions[i];
            velocities[i] = 0;
        }

        for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
            gBest[k] = pBests[k];

        clock_t begin = clock();
        cuda_pso(positions, velocities, pBests, gBest);    
        clock_t end = clock();
        printf("GPU \t ");
        printf("%10.3lf \t", (double)(end - begin)/CLOCKS_PER_SEC);
        
        printf(" %f\n", host_fitness_function(gBest));

    return 0;
}


### 1.2 — Compiler et exécuter
`-arch=sm_75` = architecture du T4. Sans ça, selon la version de CUDA installée sur Colab, il arrive que les kernels ne soient **pas lancés du tout, sans aucun message d'erreur** (d'où l'intérêt du contrôle d'erreurs qu'on ajoutera plus tard).

Le warning sur `math_functions.h` est normal (header interne déprécié).

In [ ]:
!nvcc -O2 -arch=sm_75 -o pso main.cpp kernel.cpp kernel.cu
!./pso

**Note le temps et le minimum obtenus.** Fonction sélectionnée = 0 (Levy 3D), 512 particules, 3 dimensions.

### 1.3 — Comprendre l'architecture du code

| Fichier | Rôle | S'exécute sur |
|---|---|---|
| `kernel.h` | constantes (`NUM_OF_PARTICLES`, `NUM_OF_DIMENSIONS`, `MAX_ITER`, ω, c1, c2) + prototypes | — |
| `main.cpp` | initialise positions/vitesses aléatoires, appelle `cuda_pso`, mesure le temps | CPU (host) |
| `kernel.cpp` | `host_fitness_function`, générateurs aléatoires | CPU |
| `kernel.cu` | `fitness_function` (`__device__`), 2 kernels (`__global__`), et `cuda_pso` qui orchestre | GPU + CPU |

**Organisation des données** : la population est stockée dans un **tableau 1D** de taille `N×D` : la particule `p` occupe les cases `p*D … p*D+D-1`. Un thread = **un élément** (une coordonnée d'une particule), d'où `i = blockIdx.x*blockDim.x + threadIdx.x` et `gBest[i % D]`.

**Une itération de la boucle principale :**
```
CPU ──► kernelUpdateParticle<<<>>>   (GPU : v ← ωv + c1·r1·(pBest−x) + c2·r2·(gBest−x) ; x ← x+v)
CPU ──► kernelUpdatePBest<<<>>>      (GPU : 1 thread sur D évalue f(x) et f(pBest), met à jour pBest)
GPU ──► cudaMemcpy pBests → CPU      (transfert N×D floats via PCIe)
CPU     boucle sur N particules : recalcule f(pBest) et f(gBest) pour trouver le meilleur
CPU ──► cudaMemcpy gBest → GPU
```
C'est exactement la question centrale du module (et de l'article de Qin et al.) : **où sont les transferts mémoire, et combien de threads travaillent vraiment ?**

### 1.4 — Diagnostic : ce qui ne va pas dans ce code

Je les classe par gravité. Ceux marqués ✅ ont été vérifiés par exécution, les autres par lecture du code.

**A. Bugs de correction (résultats faux)**
1. **Race condition dans `kernelUpdatePBest`** : `tempParticle1/2` sont des tableaux `__device__` **globaux, partagés par tous les threads**. Les N threads actifs écrivent tous dans le même buffer en même temps → chaque thread peut évaluer la particule d'un autre. Les mises à jour de pBest sont donc aléatoirement fausses. Correction : tableaux locaux (registres) ou évaluation directe via un pointeur `&positions[i]`.
2. ✅ **Griewank : `produit = 0`** au départ → le produit vaut toujours 0, on optimise en réalité `Σz²/4000 + 1`. Il faut `produit = 1`.
3. ✅ **`getRandom` faux** : `((high-low)+1)` → tire dans **[-5.12, 6.12)** au lieu de [-5.12, 5.12]. Population initiale biaisée.
4. **Pas de décalage (shift)** : `zi = x[i] - 0` → o = 0. Ce ne sont donc pas des fonctions « shifted ». Avec un domaine symétrique, l'optimum au centre favorise artificiellement l'algorithme. Il faut un vecteur `o` aléatoire.
5. **Domaines de recherche** : tout est dans [-5.12, 5.12]. Les fonctions de référence (CEC 2005, citées par l'article) utilisent Sphere [-100,100], Rosenbrock [-100,100], Griewank [-600,600], Rastrigin [-5,5].
6. **Aucune gestion des bornes** : les particules peuvent sortir du domaine.

**B. Protocole non respecté (par rapport au sujet)**

7. **Budget d'évaluations** : le sujet dit *max FE = 10⁴ × D*. Ici `MAX_ITER = 10⁴ × D` **itérations**, chacune évaluant 512 particules → **512 fois trop d'évaluations** (15,4 M au lieu de 30 000 en 3D). Il faut `max_iter = maxFE / N`.
8. **D et N fixés à la compilation** : `main` lit `argv` mais les variables `dim`/`pop` meurent dans le `if` et ne servent à rien. Le sujet exige que l'utilisateur choisisse D et la population.
9. ✅ **Précision float et biais** : on affiche f(x) avec les biais (−450, 390…). Autour de 450, l'écart entre deux float consécutifs est **3·10⁻⁵** : impossible de mesurer une erreur < 10⁻⁸ (critère de succès de l'article). On calculera l'**erreur** f(x) − f(x*) **sans** ajouter le biais, et/ou en `double`.

**C. Performance (le cœur du module)**

10. **gBest calculé sur le CPU à chaque itération** : copie D→H de toute la population + 2N évaluations CPU + copie H→D. Le « temps GPU » mesuré est surtout du PCIe et du CPU. À remplacer par une **réduction parallèle** sur GPU.
11. **Occupation** : dans `kernelUpdatePBest`, seul 1 thread sur D travaille (`i % D == 0`), les autres sortent immédiatement → divergence et faible occupation.
12. `blocksNum = ceil(size / threadsNum)` : division **entière** avant `ceil`. Ça marche ici par chance (1536/32 = 48 pile), sinon les derniers éléments ne sont jamais traités. Correct : `(size + T - 1) / T`.
13. **Aléatoire** : `r1, r2` sont tirés **une seule fois par itération pour toute la population** (PSO standard : un tirage par particule et par dimension). Solution : **cuRAND** sur le GPU (c'est ce que fait cudaDEi).
14. `clock()` mesure le temps CPU du processus, pas le temps réel ; on utilisera `cudaEvent` ou `std::chrono`.

**D. Mineur** : `phi = 3.1415` au lieu de π ; la formule de Levy est fausse (le terme `(yn−1)²` est ajouté D−1 fois, et il manque le facteur sin²(2πw_d)), mais Levy ne fait pas partie du projet.

### 1.5 — Preuve par l'exécution (bugs 2, 3, 7, 9) — code CPU pur

In [ ]:
%%writefile verif.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
float getRandom(float low, float high){ return low + float(((high-low)+1)*rand()/(RAND_MAX+1.0)); } // version du prof
int main(){
  float mn=1e9f, mx=-1e9f;
  for(int i=0;i<1000000;i++){ float r=getRandom(-5.12f,5.12f); mn=fminf(mn,r); mx=fmaxf(mx,r); }
  printf("[Bug 3] getRandom(-5.12,5.12) donne des valeurs dans [%f, %f]\n", mn, mx);

  float x[3]={1.f,2.f,3.f}, somme=0, produit=0;               // version du prof
  for(int i=0;i<3;i++){ somme+=x[i]*x[i]/4000; produit*=cosf(x[i]/sqrtf(i+1.f)); }
  float p1=1; for(int i=0;i<3;i++) p1*=cosf(x[i]/sqrtf(i+1.f)); // version correcte
  printf("[Bug 2] produit Griewank : prof = %f, correct = %f\n", produit, p1);

  printf("[Bug 9] pas du float autour de 450 : %g\n", nextafterf(450.f,1e9f)-450.f);
  printf("[Bug 7] FE réalisées = 30000 x 512 = %ld  vs budget = 30000\n", 30000L*512);
}

In [ ]:
!g++ -O2 verif.cpp -o verif && ./verif

### 1.6 — Questions de compréhension (réponds-moi dans le chat, on corrige ensemble)

1. Dans `kernelUpdateParticle`, combien de threads sont lancés en tout ? Combien travaillent vraiment ?
2. Pourquoi `gBest[i % NUM_OF_DIMENSIONS]` et pas `gBest[i]` ?
3. Si on passe à D = 100 et N = 500, combien de `cudaMemcpy` par itération, et de quelle taille (en octets) ?
4. Pourquoi une race condition ne fait-elle pas forcément planter le programme, et pourquoi c'est pire qu'un plantage ?
5. Avec le budget correct (10⁴·D FE), combien d'itérations fait-on pour D = 10 et N = 50 ? Et pour N = 500 ? Qu'est-ce que ça implique sur la qualité de la solution ?

**Étape suivante (2)** : écrire le **DE séquentiel propre en C++** (DE/rand/1/bin, F = 0.5, CR = 0.3 comme dans l'article), avec les 4 fonctions corrigées. C'est notre référence de correction *et* de temps (le « sDE » de l'article).

### Fin de séance — sauvegarder sur GitHub
Ne versionne que le code et les résultats utiles, jamais les exécutables (le `.gitignore` s'en charge).

In [ ]:
%cd /content/DE_CUDA
!git add -A
!git status --short
!git commit -m "Etape 1 : execution du PSO original + verifications" && git push